# Social Content to Donation Impact Optimization

## 1. Problem Framing
- **Business question:** Which social post characteristics drive donation referrals and donation value?
- **Predictive goal:** estimate expected referrals/value before posting.
- **Explanatory goal:** quantify associations between content choices and donation outcomes.

## 2. Data Acquisition, Preparation, and Exploration
- **Source used:** `social_media_posts` table from `lighthouse_csv_v7`.
- **Preparation:** target variables (`donation_referrals`, `estimated_donation_value_php`) are defined, leakage-prone post-publication engagement fields are excluded, and publish-time features are retained.
- **Temporal split:** chronological train/test split preserves realistic forecasting conditions.
- **Exploration checks:** outcome distributions, feature completeness, and category balance are reviewed to guide model and transform decisions.

## 3. Modeling and Feature Selection
- **Explanatory model:** Linear Regression for directionally interpretable relationships.
- **Predictive model:** Random Forest Regressor for stronger non-linear fit.
- **Feature rationale:** selected inputs are controllable pre-publish levers (timing, CTA, topic, format, budget) that can inform content planning decisions.

## Run Instructions
- Run from repository root so `Path('lighthouse_csv_v7')` resolves correctly.
- Install dependencies in `requirements.txt` before execution.
- If data is missing, follow the explicit path guidance from the first code cell error message.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

DATA = Path('lighthouse_csv_v7')
if not DATA.exists():
    raise FileNotFoundError(
        "Missing data folder 'lighthouse_csv_v7'. Run notebook from repo root and place CSVs at ./lighthouse_csv_v7/."
    )

posts = pd.read_csv(DATA / 'social_media_posts.csv', parse_dates=['created_at'])
posts = posts.sort_values('created_at').copy()

posts['target_referrals'] = posts['donation_referrals'].fillna(0)
posts['target_value'] = posts['estimated_donation_value_php'].fillna(0)

# Leakage checklist:
# Use only fields known at publish time. Exclude post-performance fields like reach/likes/views.
feature_cols = [
    'platform','day_of_week','post_hour','post_type','media_type','num_hashtags',
    'mentions_count','has_call_to_action','call_to_action_type','content_topic','sentiment_tone',
    'caption_length','is_boosted','boost_budget_php'
]

df = posts[['created_at'] + feature_cols + ['target_referrals','target_value']].copy()

cut = int(len(df) * 0.8)
train_df = df.iloc[:cut].copy()
test_df = df.iloc[cut:].copy()

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
X = df[feature_cols]
y_ref = df['target_referrals']
y_val = df['target_value']

num_cols = X_train.select_dtypes(include=['number','bool']).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

def run_regression(y_col, label, log_target=False):
    y_train = train_df[y_col]
    y_test = test_df[y_col]

    baseline = np.repeat(float(y_train.mean()), len(y_test))
    print(label, 'Baseline_Mean', 'MAE', round(mean_absolute_error(y_test, baseline), 2), 'R2', round(r2_score(y_test, baseline), 3))

    exp_model = Pipeline([('pre', pre), ('reg', LinearRegression())])
    pred_model = Pipeline([('pre', pre), ('reg', RandomForestRegressor(n_estimators=400, random_state=42))])

    if log_target:
        y_train_fit = np.log1p(y_train)
    else:
        y_train_fit = y_train

    for name, model in [('Explanatory_Linear', exp_model), ('Predictive_RF', pred_model)]:
        model.fit(X_train, y_train_fit)
        p = model.predict(X_test)
        if log_target:
            p = np.expm1(p)
        print(label, name + ('_LogTarget' if log_target else ''), 'MAE', round(mean_absolute_error(y_test, p), 2), 'R2', round(r2_score(y_test, p), 3))

run_regression('target_referrals', 'DonationReferrals', log_target=False)
run_regression('target_value', 'DonationValuePHP', log_target=True)


In [ ]:
# EDA evidence: missingness, target distributions, and feature summaries
print('Missingness ratio (top 10):')
print(df.isna().mean().sort_values(ascending=False).head(10).to_string())
print('-' * 70)

print('Referral target summary:')
print(df['target_referrals'].describe().to_string())
print('-' * 70)
print('Donation value target summary:')
print(df['target_value'].describe().to_string())
print('-' * 70)

num_eda_cols = [c for c in feature_cols if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
print('Numeric feature summary:')
print(df[num_eda_cols].describe().to_string())

## 4. Evaluation and Interpretation
- **Validation strategy:** chronological holdout evaluation with baseline comparison and cross-validated model selection.
- **Metrics used:** MAE and R2 for both referrals and donation value to quantify operational forecast error and explained variance.
- **Error impact:** underprediction can underfund high-impact campaigns; overprediction can misallocate media budget.
- **Threshold/selection policy:** choose model with lowest MAE when budget planning accuracy is the primary operational objective.

## 5. Causal and Relationship Analysis
- CTA type, content topic, boost budget, and timing often show strong associations with fundraising outcomes.
- **What this explains:** which pre-publish content choices are most associated with stronger referral/value outcomes.
- **What this cannot claim:** causal marketing lift without controlled A/B experimentation.
- Recommended decisions: prioritize high-performing CTA/topic/time combinations and use projected value to guide boost spending.

## 6. Deployment Notes
- Endpoint: `POST /api/ml/social-post-impact`.
- UI: pre-publish planner estimates expected referrals and donation value for draft posts.
- Current status: model scoring is deployed; database storage for prediction logs is planned for a later iteration.
- Repo deployment map: see `pipeline/DEPLOYMENT_MAP.md` for endpoint/UI file references and verification checklist.

In [ ]:
# Model selection for social content impact (referrals + donation value)
from sklearn.model_selection import KFold, cross_validate
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

candidate_regressors = {
    'LinearRegression': LinearRegression(),
    'DecisionTreeRegressor': DecisionTreeRegressor(random_state=42),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=300, random_state=42),
    'GradientBoostingRegressor': GradientBoostingRegressor(random_state=42)
}

cv_splits = min(5, len(X))
cv_splits = max(2, cv_splits)
cv = KFold(n_splits=cv_splits, shuffle=True, random_state=42)
scoring = {'mae': 'neg_mean_absolute_error', 'r2': 'r2'}

def select_best_regressor(y, label):
    rows = []
    for model_name, estimator in candidate_regressors.items():
        pipe = Pipeline([('pre', pre), ('reg', estimator)])
        scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1, error_score='raise')
        rows.append({
            'model': model_name,
            'cv_folds': cv_splits,
            'mae_mean': -scores['test_mae'].mean(),
            'r2_mean': scores['test_r2'].mean()
        })
    out = pd.DataFrame(rows).sort_values('mae_mean', ascending=True)
    print(f'Model selection results ({label}):')
    print(out.to_string(index=False))
    print('-' * 70)
    return out

referral_model_selection = select_best_regressor(y_ref, 'Donation Referrals')
value_model_selection = select_best_regressor(y_val, 'Donation Value PHP')

In [ ]:
# Feature-selection evidence: explanatory coefficients and predictive importances
exp_ref = Pipeline([('pre', pre), ('reg', LinearRegression())])
exp_ref.fit(X_train, train_df['target_referrals'])
feature_names = exp_ref.named_steps['pre'].get_feature_names_out()
coef_vals = exp_ref.named_steps['reg'].coef_
coef_table = pd.DataFrame({'feature': feature_names, 'coef': coef_vals})
coef_table['abs_coef'] = coef_table['coef'].abs()
print('Top explanatory coefficients (referrals):')
print(coef_table.sort_values('abs_coef', ascending=False).head(10)[['feature','coef']].to_string(index=False))
print('-' * 70)

pred_ref = Pipeline([('pre', pre), ('reg', RandomForestRegressor(n_estimators=400, random_state=42))])
pred_ref.fit(X_train, train_df['target_referrals'])
imp_vals = pred_ref.named_steps['reg'].feature_importances_
imp_table = pd.DataFrame({'feature': feature_names, 'importance': imp_vals})
print('Top predictive importances (referrals):')
print(imp_table.sort_values('importance', ascending=False).head(10).to_string(index=False))